<a href="https://colab.research.google.com/github/Anna94652/10701-Secondary-Protein-Structure-Prediction/blob/main/701ProjectDataPreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install huggingface_hub -q
from huggingface_hub import hf_hub_download
import pandas as pd

# Training set
train_path = hf_hub_download(
    repo_id="proteinea/secondary_structure_prediction",
    filename="training_hhblits.csv",
    repo_type="dataset"
)
train_df = pd.read_csv(train_path)
print(train_df.shape)
print(train_df.columns.tolist())

# Test set (CB513)
test_path = hf_hub_download(
    repo_id="proteinea/secondary_structure_prediction",
    filename="CB513.csv",
    repo_type="dataset"
)
test_df = pd.read_csv(test_path)
print(test_df.shape)

training_hhblits.csv:   0%|          | 0.00/30.4M [00:00<?, ?B/s]

(10792, 5)
['input', 'dssp3', 'dssp8', 'disorder', 'cb513_mask']
(511, 5)


In [ ]:
# See all unique characters actually present in the labels
all_chars = set(''.join(train_df['dssp8'].dropna()))
print(sorted(all_chars))

['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T']


In [ ]:
import numpy as np
import pandas as pd

# Amino acid vocabulary (same order as original dataset)
AA_VOCAB = 'ACDEFGHIKLMNPQRSTVWXY'
SS_VOCAB  = 'CEHT'   # dssp3: Coil, hElix, sTrand (+ T for rare cases)

# Or for 8-class:
SS8_VOCAB = 'CBEGIHST'

MAX_LEN = 700  # pad/truncate all sequences to this length

In [ ]:
def encode_sequence(seq, vocab, max_len):
    vocab_map = {c: i for i, c in enumerate(vocab)}
    n = len(vocab)
    arr = np.zeros((max_len, n), dtype=np.float32)
    for i, ch in enumerate(seq[:max_len]):
        if ch in vocab_map:
            arr[i, vocab_map[ch]] = 1.0
    return arr   # shape (max_len, n)

In [ ]:
def encode_df(df):
    X, y = [], []
    for _, row in df.iterrows():
        X.append(encode_sequence(row['input'], AA_VOCAB, MAX_LEN))
        y.append(encode_sequence(row['dssp8'], 'CBEGIHST', MAX_LEN))
    return np.array(X), np.array(y)

X_train, y_train = encode_df(train_df)
X_test,  y_test  = encode_df(test_df)

print(X_train.shape)  # (N, 700, 21)
print(y_train.shape)  # (N, 700, 3)

(10792, 700, 21)
(10792, 700, 8)


In [ ]:
def make_mask(df, max_len=MAX_LEN):
    masks = []
    for _, row in df.iterrows():
        length = min(len(row['input']), max_len)
        mask = np.zeros(max_len, dtype=np.float32)
        mask[:length] = 1.0
        masks.append(mask)
    return np.array(masks)   # shape (N, 700)

train_mask = make_mask(train_df)
test_mask  = make_mask(test_df)

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [ ]:
import pandas as pd

AA_VOCAB = list('ACDEFGHIKLMNPQRSTVWXY')
SS_VOCAB = list('CHE')
SS8_VOCAB = list('CBEGIHST')

def view_protein(X, y, idx):
    # Find where the sequence ends (all zeros = padding)
    length = int((X[idx].sum(axis=1) > 0).sum())

    aa_df  = pd.DataFrame(X[idx, :length, :], columns=AA_VOCAB)
    ss_df  = pd.DataFrame(y[idx, :length, :], columns=SS8_VOCAB)

    # Decode back to letters for readability
    aa_df['amino_acid'] = aa_df.idxmax(axis=1)
    ss_df['structure']  = ss_df.idxmax(axis=1)

    return pd.concat([aa_df[['amino_acid']], ss_df[['structure']]], axis=1)

view_protein(X_train, y_train, 10791)

,amino_acid,structure
0,E,C
1,N,C
2,L,E
3,K,E
4,L,E
...,...,...
301,G,T
302,L,C
303,G,C
304,G,C


In [ ]:
# One row per protein — sequence length and most common structure
SS8_VOCAB = ['C', 'B', 'E', 'G', 'I', 'H', 'S', 'T']

mask = (X_train.sum(axis=2) > 0)  # (N, 700) — True where real residue

summary = pd.DataFrame(
    {f'pct_{ss}': (y_train[:, :, i] * mask).sum(axis=1) / mask.sum(axis=1)
     for i, ss in enumerate(SS8_VOCAB)},
)
summary.insert(0, 'length', mask.sum(axis=1))

print(summary.describe())

             length         pct_C         pct_B         pct_E         pct_G  \
count  10792.000000  10792.000000  10792.000000  10792.000000  10792.000000   
mean     252.218495      0.252888      0.008429      0.202000      0.033794   
std      147.637404      0.096839      0.010516      0.147289      0.026596   
min       20.000000      0.009709      0.000000      0.000000      0.000000   
25%      142.000000      0.195306      0.000000      0.100250      0.015075   
50%      218.000000      0.238558      0.006006      0.191382      0.031579   
75%      333.000000      0.290000      0.013583      0.294118      0.049261   
max      700.000000      1.000000      0.229167      0.750000      0.409091   

              pct_I         pct_H         pct_S         pct_T  
count  10792.000000  10792.000000  10792.000000  10792.000000  
mean       0.004411      0.329370      0.070820      0.098288  
std        0.011318      0.201758      0.032774      0.036877  
min        0.000000      0.00000

In [ ]:
print(train_df.isnull().sum())
print(train_df['input'].str.len().describe())  # sequence length distribution

input         0
dssp3         0
dssp8         0
disorder      0
cb513_mask    0
dtype: int64
count    10792.000000
mean       255.796238
std        161.926369
min         20.000000
25%        142.000000
50%        218.000000
75%        333.000000
max       1632.000000
Name: input, dtype: float64


In [ ]:
train_sequences = set(train_df['input'].str.strip())
test_sequences  = set(test_df['input'].str.strip())

overlap = train_sequences & test_sequences
print(f"Training proteins:       {len(train_sequences)}")
print(f"Test proteins (CB513):   {len(test_sequences)}")
print(f"Overlapping sequences:   {len(overlap)}")

Training proteins:       10792
Test proteins (CB513):   432
Overlapping sequences:   2


In [ ]:
train_df_clean = train_df[~train_df['input'].str.strip().isin(test_sequences)]
print(f"Training set after removing overlap: {len(train_df_clean)}")

Training set after removing overlap: 10790


In [ ]:
# Normalise fully before comparing
train_df['input_clean'] = train_df['input'].str.strip().str.upper()
test_df['input_clean']  = test_df['input'].str.strip().str.upper()

overlap = set(train_df['input_clean']) & set(test_df['input_clean'])
print(f"Overlap after normalisation: {len(overlap)}")

Overlap after normalisation: 2
